In [1]:
import numpy as np
import cv2


In [ ]:
import cv2
import numpy as np

# -----------------------------
# Open Camera
# -----------------------------
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot open camera.")
    exit()

# -----------------------------
# Read First Frame
# -----------------------------
ret, frame = cap.read()

if not ret:
    print("Error: Cannot read frame.")
    cap.release()
    exit()

# -----------------------------
# Load Haar Cascade
# -----------------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("Error: Haar Cascade not loaded.")
    cap.release()
    exit()

# -----------------------------
# Detect Face
# -----------------------------
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
gray = cv2.equalizeHist(gray)

faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(80, 80)
)

if len(faces) == 0:
    print("No face detected.")
    cap.release()
    cv2.destroyAllWindows()
    exit()

# Select the largest detected face
faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)
(face_x, face_y, w, h) = faces[0]

track_window = (face_x, face_y, w, h)

# -----------------------------
# ROI for MeanShift
# -----------------------------
roi = frame[face_y:face_y+h, face_x:face_x+w]

hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

mask = cv2.inRange(
    hsv_roi,
    np.array((0., 60., 32.)),
    np.array((180., 255., 255.))
)

roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])

cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Termination criteria
term_crit = (
    cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
    10,
    1
)

# -----------------------------
# Tracking Loop
# -----------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    dst = cv2.calcBackProject(
        [hsv],
        [0],
        roi_hist,
        [0, 180],
        1
    )

    ret, track_window = cv2.meanShift(
        dst,
        track_window,
        term_crit
    )

    x, y, w, h = track_window

    cv2.rectangle(
        frame,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        3
    )

    cv2.imshow("MeanShift Face Tracking", frame)

    key = cv2.waitKey(30) & 0xFF

    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

No face detected.


IndexError: list index out of range

: 